# Part 13 — Late-Interaction Retrieval

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mftnakrsu/rag-by-hand/blob/main/part-13-late-interaction/late_interaction.ipynb)

*RAG from First Principles. Part 13 of the series, opening the Frontier Track.*

📖 Read the essay: https://www.mefby.com/essays/late-interaction-retrieval

🐍 Companion script: `late_interaction.py`

## What you'll build

The core series ended at Part 12. This is the **Frontier Track** — optional, 2026-frontier material you reach for once the core is solid.

Back in **Part 7** every passage became **one** dense vector. In **Part 8** a cross-encoder re-scored a query against each candidate with the full model — high quality, but it must run that model on every query-doc pair *at query time*. **Late interaction** sits between them: keep a vector **per token** for both the query and the document, and score a pair with **MaxSim** — for each query token, take its max similarity over all doc tokens, then sum. You get cross-encoder-style fine-grained term matching, but the document vectors are **precomputed offline**, so serving stays cheap. *"Cross-encoder quality, bi-encoder serving cost."*

Concretely, by the end you'll have built, by hand and in numpy:

- a `token_embed(text)` that returns a vector **per token** (not one pooled vector);
- `maxsim(query_vecs, doc_vecs)` — the late-interaction score, three lines of numpy;
- a head-to-head ranking, **MaxSim vs pooled cosine**, on a term-match query, where the two **disagree**;
- the **storage arithmetic** (per-token vectors cost more; ColBERTv2 residual compression brings it back down);
- a toy **ColPali patch-vector** stand-in, to make the page-image mechanism concrete with the *same* MaxSim.

> **Runs fully offline.** Every code cell executes with **no network and no API key** — numpy only. The real `sentence-transformers` token-embedding path is the headline (the code you'd actually ship); if a model is unavailable, a transparent deterministic hashing stand-in keeps the cell runnable and prints a clear label when it kicks in. **ColPali is taught strictly as a mechanism with a toy stand-in: no real vision-language model runs offline here.**

Citations: ColBERT (Khattab & Zaharia, SIGIR 2020, arXiv 2004.12832); ColBERTv2 (Santhanam et al., NAACL 2022, arXiv 2112.01488); ColPali (Faysse et al., ICLR 2025, arXiv 2407.01449).

## Setup

Two imports for the math (`re` to tokenize, `numpy` for the vectors) and `hashlib` for the offline stand-in. Nothing here touches the network.

In [ ]:
import hashlib
import re

import numpy as np

print("numpy", np.__version__, "— ready (no network, no API key required)")

## Step 0 — The running example

We reuse the support knowledge base from Parts 6–12, but arranged to expose the **pooling problem**. The query is a **term-match** query, `"E-4042 error"`. The corpus holds:

- an **exact-term** doc where the rare code `E-4042` is **buried** deep inside a long, mostly-unrelated passage (the way a real passage would carry it);
- a short, generic **distractor** that shares the topical word *"error"* but **not** the rare code;
- two off-topic policy lines (shipping, refunds).

This is exactly the case where a single pooled vector struggles: averaging a long passage washes the rare token out, and the short generic doc pools higher. Late interaction's per-token MaxSim catches the exact hit.

In [ ]:
CORPUS = [
    # Exact-term doc: E-4042 buried inside a long, mostly-unrelated passage.
    "Our internal billing log emits the diagnostic identifier E-4042 deep inside a long "
    "batch reconciliation report alongside dozens of unrelated routine status lines.",
    # Distractor: short and generic, shares "error" but not the rare code.
    "A general error occurred while loading the page.",
    "Standard shipping takes three to five business days to arrive at the address on the order.",
    "Refunds are accepted within 30 days of purchase if the item is unused and in its packaging.",
]
QUERY = "E-4042 error"

for i, doc in enumerate(CORPUS):
    print(f"[{i}] {doc[:70]}...")
print(f'\nQuery: "{QUERY}"')

## Step 1 — Tokenize

We stay at the level of plain lowercase word/number tokens. Note that `E-4042` splits into `e` and `4042` — which is exactly *why* a rare-term query can still find an exact token match downstream: the literal `4042` token reappears, untouched, in the right document.

In [ ]:
def tokenize(text):
    """Lowercase word/number tokens — the level the essay stays at."""
    return re.findall(r"[a-z0-9]+", text.lower())


print(tokenize(QUERY))                       # ['e', '4042', 'error']
print(tokenize(CORPUS[0])[:8], "...")        # the exact-term doc's first tokens

## Step 2 — A vector **per token**, with a transparent offline fallback

A single-vector model (Part 7) pools the encoder's per-token states into **one** vector. A late-interaction model (ColBERT) keeps the **per-token** hidden states instead. With `sentence-transformers` we can pull those token embeddings straight out of the underlying transformer.

This is the **headline** — the code you'd actually ship. We wrap the load in `try/except` so the notebook still runs with no model and no network: on failure we drop to a transparent deterministic stand-in (next cell), and a banner says which path is live.

In [ ]:
def load_real_token_embedder():
    """Return token_embed(text)->(n_tokens, d) from a real model, or None offline."""
    try:
        from sentence_transformers import SentenceTransformer

        model = SentenceTransformer("all-MiniLM-L6-v2")     # 384 dims

        def token_embed(text):
            # Pull the transformer's per-token hidden states (before pooling).
            features = model.tokenize([text])
            features = {k: v.to(model.device) for k, v in features.items()}
            out = model.forward(features)
            tok = out["token_embeddings"][0].detach().cpu().numpy()   # (seq, d)
            mask = features["attention_mask"][0].detach().cpu().numpy().astype(bool)
            tok = tok[mask]                                           # drop padding
            norms = np.linalg.norm(tok, axis=1, keepdims=True)        # L2-normalize
            norms[norms == 0] = 1.0
            return tok / norms

        return token_embed
    except Exception as exc:           # missing package / no network / no weights
        print(f"  [real model unavailable: {type(exc).__name__}] -> using offline fallback")
        return None


print("load_real_token_embedder defined.")

### The offline stand-in: hash each token to a unit vector

A real late-interaction model learned, from billions of sentences, to place token meanings sensibly in space; we can't reproduce that without the weights. What we **can** do, with zero dependencies, is mimic the **interface**: any token → a fixed-length dense unit vector, deterministic and comparable in one space.

The key property for this lesson: **identical tokens map to the same vector**. So the query's `4042` and the exact-term doc's `4042` line up perfectly (cosine ≈ 1.0), making MaxSim's fine-grained matching visible — while being honest that a hash has no idea *"refund"* and *"reimbursement"* are cousins.

In [ ]:
FALLBACK_DIM = 384    # mirror all-MiniLM-L6-v2's 384 dims on purpose.


def _hashed_token_vector(token, dim=FALLBACK_DIM):
    # Deterministic hash -> a reproducible pseudo-random direction for this token.
    h = hashlib.sha256(token.encode("utf-8")).digest()
    raw = np.frombuffer((h * (dim // len(h) + 1))[:dim], dtype=np.uint8)
    vec = (raw.astype(np.float64) / 255.0) * 2.0 - 1.0
    norm = np.linalg.norm(vec)
    return vec if norm == 0 else vec / norm


def fallback_token_embed(text, dim=FALLBACK_DIM):
    """Any text -> (n_tokens, dim) of L2-normalized per-token vectors."""
    tokens = tokenize(text)
    if not tokens:
        return np.zeros((1, dim))
    return np.vstack([_hashed_token_vector(t, dim) for t in tokens])


# Identical tokens -> identical vectors -> cosine 1.0 (the property MaxSim leans on):
a = _hashed_token_vector("4042")
b = _hashed_token_vector("4042")
print(f'cosine("4042", "4042") = {float(a @ b):.3f}   (exact term match)')

Now pick the live embedder: the real model if it loads, the stand-in otherwise. The banner always names the active path.

In [ ]:
_real = load_real_token_embedder()
using_real = _real is not None
token_embed = _real if using_real else fallback_token_embed
print("Embedder in use:",
      "REAL all-MiniLM-L6-v2 token vectors" if using_real
      else "offline hashing fallback (deterministic, numpy only)")

demo = token_embed("Refund requests are processed within five business days.")
print(f"token_embed returns {demo.shape[0]} vectors x {demo.shape[1]} dims "
      "— one vector PER TOKEN, nothing averaged away.")

## Step 3 — MaxSim, the late-interaction score

Here is the whole idea in three lines. The query is `(n_q, d)`, the doc is `(n_d, d)`, both L2-normalized so a dot product **is** cosine similarity. We form the full `(n_q, n_d)` matrix of pairwise cosines, take the **max over doc tokens** for each query token (each query term finds its single best match in the doc), and **sum** those maxima over the query tokens.

That `max` is what rewards a *single strong hit* — the exact `4042` match — even when it is buried in a long passage. A pooled vector would have averaged it away.

In [ ]:
def maxsim(query_vecs: np.ndarray, doc_vecs: np.ndarray) -> float:
    """Late-interaction score: for each query token take the max cosine
    similarity over all doc tokens, then sum. Inputs are L2-normalized
    (n_q, d) and (n_d, d), so the dot product IS cosine similarity."""
    sims = query_vecs @ doc_vecs.T          # (n_q, n_d) pairwise cosine
    return float(sims.max(axis=1).sum())    # max over doc tokens, sum over query tokens


print("maxsim() ready.")

## Step 4 — The pooled-cosine baseline (Part 7), to compare against

Late interaction is competing against the single-vector approach from Part 7: mean the token vectors into **one** vector per text, then compare with a single cosine. We build that baseline so the next step can show the two methods **disagree**.

In [ ]:
def pool(token_vecs):
    """Mean-pool per-token vectors into one passage vector, L2-normalized."""
    v = token_vecs.mean(axis=0)
    norm = np.linalg.norm(v)
    return v if norm == 0 else v / norm


def pooled_cosine(query_vecs, doc_vecs):
    """One pooled query vector vs one pooled doc vector (Part 7 single-vector)."""
    return float(np.dot(pool(query_vecs), pool(doc_vecs)))


print("pool() and pooled_cosine() ready.")

## Step 5 — Head-to-head: MaxSim vs pooled cosine

Embed the query and every doc once, then rank the corpus both ways. Watch the **rank-1 disagreement**: pooled cosine puts the short generic *"error"* **distractor** on top (its few tokens are dominated by the topical word), and buries the exact-term doc; MaxSim surfaces the **exact-term** doc, because the query's `4042` token finds its exact match there.

In [ ]:
q_vecs = token_embed(QUERY)
doc_vecs = [token_embed(doc) for doc in CORPUS]


def tag(doc):
    if "E-4042" in doc:
        return "[exact term]"
    if doc.startswith("A general error"):
        return "[DISTRACTOR]"
    return ""


pooled_ranked = sorted(
    ((pooled_cosine(q_vecs, dv), doc) for dv, doc in zip(doc_vecs, CORPUS)),
    key=lambda pair: -pair[0],
)
print("Ranking by POOLED cosine (one vector per doc, Part 7):")
for rank, (score, doc) in enumerate(pooled_ranked, 1):
    print(f"  rank {rank}  cos={score:+.3f}  {doc[:52]}... {tag(doc)}".rstrip())
print("  -> the exact-term doc is NOT at the top: pooling averaged the rare token away.")

maxsim_ranked = sorted(
    ((maxsim(q_vecs, dv), doc) for dv, doc in zip(doc_vecs, CORPUS)),
    key=lambda pair: -pair[0],
)
print("\nRanking by MaxSim (token-level late interaction):")
for rank, (score, doc) in enumerate(maxsim_ranked, 1):
    print(f"  rank {rank}  maxsim={score:.3f}  {doc[:52]}... {tag(doc)}".rstrip())
print('  -> late interaction surfaces the exact-term doc: one strong "4042" hit wins.')

## Step 6 — Why it's still cheap to serve

The doc multi-vectors above were computed **once**, ahead of time. At query time only `maxsim` runs — pure numpy over precomputed matrices, no model. Contrast Part 8's cross-encoder, which must run the full model on **every** query-doc pair at query time. That's the whole trade: *cross-encoder quality, bi-encoder serving cost.*

In [ ]:
# Precompute the doc vectors ONCE (offline / index time)...
doc_index = [token_embed(doc) for doc in CORPUS]

# ...then at QUERY time, only cheap numpy runs — no model call:
q = token_embed("E-4042 error")
scores = [maxsim(q, dv) for dv in doc_index]
print("query-time work = len(query_tokens) x sum(doc_tokens) dot products, all numpy")
print("top doc by MaxSim:", CORPUS[int(np.argmax(scores))][:60], "...")

## Step 7 — The storage tradeoff

Nothing is free: a vector **per token** costs roughly `avg_tokens` times more than one pooled vector. **ColBERTv2** addresses this with **residual compression** (1–2 bits per dimension). We compute the toy numbers, then quote the paper's MS MARCO scale for context: the vanilla multi-vector index is **~154 GiB**, shrunk to **~16 GiB (1-bit) / ~25 GiB (2-bit)** by residual compression — a 6–10x reduction.

In [ ]:
def storage_report(n_docs, avg_tokens, dim, bytes_per_dim=4, compress_bits=2):
    pooled = n_docs * dim * bytes_per_dim
    multi = n_docs * avg_tokens * dim * bytes_per_dim
    multi_c = int(n_docs * avg_tokens * dim * (compress_bits / 8.0))
    print(f"Toy corpus: {n_docs} docs, ~{avg_tokens} tokens/doc, {dim} dims, float32.")
    print(f"  pooled (1 vec/doc)      : {pooled:>10,} bytes")
    print(f"  multi-vector (1 vec/tok): {multi:>10,} bytes  -> {multi/pooled:.1f}x more")
    print(f"  multi-vector, {compress_bits}-bit     : {multi_c:>10,} bytes  "
          "-> ColBERTv2 residual compression")
    print("MS MARCO scale (ColBERTv2 paper): ~154 GiB vanilla -> ~16 GiB (1-bit) / "
          "~25 GiB (2-bit), a 6-10x cut.")


storage_report(n_docs=len(CORPUS), avg_tokens=12, dim=q.shape[1])

## Step 8 — ColPali / ColQwen, as a **mechanism**

**ColPali** (Faysse et al., ICLR 2025) pushes late interaction onto document **page images**. A vision-language model embeds a *rendered page* into ~1024 **patch** vectors; the query tokens are scored against those patches with the **same MaxSim** — no OCR, no chunking, no layout parsing. A figure-heavy PDF page just becomes a bag of patch vectors.

> **No real vision-language model runs offline here.** To make the mechanism concrete we stand in a toy grid of hashed *patch* vectors for a page image's patches, and run the *identical* `maxsim` call. The point is purely that the **scoring is the same**; the doc tokens are simply image patches instead of text tokens.

In [ ]:
def colpali_patch_demo(query="refund window policy", grid=4, dim=FALLBACK_DIM):
    # Toy "page": grid*grid patch vectors, hashed from (row, col) position so the
    # demo is reproducible. A STAND-IN for an image encoder's patch embeddings.
    patches = np.vstack(
        [_hashed_token_vector(f"patch_{r}_{c}", dim)
         for r in range(grid) for c in range(grid)]
    )
    q_vecs = token_embed(query)
    score = maxsim(q_vecs, patches)
    print(f"toy page: {patches.shape[0]} patch vectors x {patches.shape[1]} dims "
          "(standing in for an image's patches)")
    print(f'query "{query}" ({q_vecs.shape[0]} tokens) vs the page patches:')
    print(f"  maxsim(query tokens, page patches) = {score:.3f}")
    print("  -> identical to text late interaction: each query token takes its max")
    print("     over the patch vectors, then we sum. ColPali just swaps text tokens")
    print("     for image patches.")


colpali_patch_demo()

## What you learned

- A single pooled vector (Part 7) averages a passage into **one** point, so a query term that matches **one word** in a long passage gets washed out — you saw the short generic *"error"* distractor out-rank the buried-`E-4042` doc under pooled cosine.
- **Late interaction** keeps a vector **per token** and scores with **MaxSim**: for each query token, max over the doc tokens, then sum. That `max` rewards a single strong, exact hit, so the right doc surfaces — on **both** the real-model and offline-fallback paths.
- It is cheap to **serve** because the doc multi-vectors are **precomputed offline**; only MaxSim runs at query time. Contrast Part 8's cross-encoder, which runs the full model on every query-doc pair at query time. *Cross-encoder quality, bi-encoder serving cost.*
- The cost shows up in **storage** (per-token vectors are many times larger); **ColBERTv2** residual compression (1–2 bits/dim) brings MS MARCO's ~154 GiB index down to ~16–25 GiB.
- **ColPali** extends the *same* MaxSim to document **page images** via patch vectors — retrieving documents without OCR or chunking. We showed the mechanism with a toy patch grid; no real VLM runs offline.
- Everything ran **fully offline**: the real `sentence-transformers` token path is the headline, with a transparent deterministic stand-in so every cell executes with no network and no API key.

### Next

**Part 14 — Context-Aware Chunking.** Late interaction fixed *scoring*; next we fix the *chunk* itself — two training-free ways (late chunking and Contextual Retrieval) to stop a chunk from losing the document context that makes it interpretable. (Previous: **Part 12 — RAG in Production**, the core finale.)